## 연습 문제 풀기 숙제 ^^!!

- 1. 3번 orm 파일에 넣고 해주세요
- 2. 1. 초기실행, 2.테이블 생성 실행하고 연습문제 풀기
- 3. 모르면 어디가 안되는지, 왜 모르는지 주석 or 마크 다운 달아놓기

## 1. 초기세팅 (실행)

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import relationship, sessionmaker, declarative_base

# 1. 엔진 연결 (기존에 만든 db 파일명 확인)
engine = create_engine('sqlite:///crime_data.db') 
Session = sessionmaker(bind=engine)
session = Session()
Base = declarative_base()

## 2. 테이블 생성 (실행)

In [ ]:
# --- [ 마스터 테이블 ] ---

class RegionMaster(Base):
    """지역 마스터 (예: 서울 강남구, 서울 종로구)"""
    __tablename__ = 'region_master'
    id = Column(Integer, primary_key=True, autoincrement=True)
    region_name = Column(String, unique=True, nullable=False)

    # 관계 설정
    mappers = relationship("RegionMapper", back_populates="region")
    region_crimes = relationship("CrimeRegion", back_populates="region")

class CrimeCategory(Base):
    """범죄 종류 마스터 (예: 절도, 폭행)"""
    __tablename__ = 'crime_category'
    id = Column(Integer, primary_key=True, autoincrement=True)
    main_cat = Column(String)  # 범죄대분류
    sub_cat = Column(String, unique=True) # 범죄중분류

    # 관계 설정
    region_stats = relationship("CrimeRegion", back_populates="category")
    time_stats = relationship("CrimeTime", back_populates="category")
    week_stats = relationship("CrimeWeek", back_populates="category")

# --- [ 매핑 테이블 (핵심 연결 고리) ] ---

class RegionMapper(Base):
    """핫스팟과 지역 마스터를 잇는 중간 다리 (mapping_fix 기반)"""
    __tablename__ = 'region_mapper'
    id = Column(Integer, primary_key=True, autoincrement=True)
    AREA_GU = Column(String)
    CATEGORY = Column(String)
    NO = Column(Integer)
    AREA_CD = Column(String)
    AREA_NM = Column(String)
    ENG_NM = Column(String)
    
    # FK 설정
    hotspot_id = Column(Integer, ForeignKey('hotspot_api.id'))
    region_id = Column(Integer, ForeignKey('region_master.id'))

    # 관계 설정
    hotspot = relationship("HotspotAPI", back_populates="mapper")
    region = relationship("RegionMaster", back_populates="mappers")

# --- [ 데이터 테이블 ] ---

class HotspotAPI(Base):
    """핫스팟 API 정보 (seoul live cache 기반)"""
    __tablename__ = 'hotspot_api'
    id = Column(Integer, primary_key=True, autoincrement=True)
    area_name = Column(String)
    congest_lvl = Column(Integer)
    ppltn_min = Column(Integer)
    ppltn_max = Column(Integer)
    temp = Column(Float)
    update_time = Column(String)
    collected_at = Column(String)

    # 매핑 테이블과 1:1 연결
    mapper = relationship("RegionMapper", back_populates="hotspot", uselist=False)

class CrimeRegion(Base):
    """경찰청 지역별 범죄 통계 (police_region_fix 기반)"""
    __tablename__ = 'crime_region'
    id = Column(Integer, primary_key=True, autoincrement=True)
    crime_count = Column(Integer)
    
    region_id = Column(Integer, ForeignKey('region_master.id'))
    category_id = Column(Integer, ForeignKey('crime_category.id'))
    
    region = relationship("RegionMaster", back_populates="region_crimes")
    category = relationship("CrimeCategory", back_populates="region_stats")

class CrimeTime(Base):
    """시간대별 범죄 통계 (전국 통계용)"""
    __tablename__ = 'crime_time'
    id = Column(Integer, primary_key=True, autoincrement=True)
    time_range = Column(String)
    crime_count = Column(Float)
    
    category_id = Column(Integer, ForeignKey('crime_category.id'))
    category = relationship("CrimeCategory", back_populates="time_stats")

class CrimeWeek(Base):
    """요일별 범죄 통계 (전국 통계용)"""
    __tablename__ = 'crime_week'
    id = Column(Integer, primary_key=True, autoincrement=True)
    day_of_week = Column(String)
    crime_count = Column(Integer)
    
    category_id = Column(Integer, ForeignKey('crime_category.id'))
    category = relationship("CrimeCategory", back_populates="week_stats")

## 3. 연습문제 - 기초 10개

### 1. RegionMaster 테이블에서 지역 이름 10개만 조회하시오.

### 2. CrimeCategory 테이블에서 범죄 중분류(sub_cat) 5개만 조회하시오

### 3. RegionMaster에서 id 기준으로 가장 첫 번째 지역 1개만 조회하시오.

### 4. CrimeCategory에서 main_cat이 "절도"인 데이터 중 상위 5개 조회하시오.

### 5. HotspotAPI 테이블에서 혼잡도(congest_lvl)가 가장 높은 데이터 1개 조회하시오.

### 6. HotspotAPI에서 temperature(temp)가 높은 순으로 상위 5개 조회하시오.

### 7. RegionMaster에서 지역 이름(region_name)을 가나다순으로 정렬 후 10개 조회하시오.

### 8. CrimeCategory에서 서로 다른 main_cat 5개 조회하시오.

### 9. HotspotAPI에서 ppltn_max 값이 높은 상위 3개 지역 조회하시오.

### 10. CrimeCategory에서 sub_cat 길이가 긴 순으로 상위 5개 조회하시오.

## 4. 연습문제 - 기본 10개

### 1. HotspotAPI에서 ppltn_min이 1000 이상인 데이터 중 상위 5개 조회하시오.

### 2. HotspotAPI에서 congest_lvl이 "여유"가 아닌 데이터 중 10개 조회하시오.

### 3. CrimeCategory에서 main_cat이 "폭력"인 범죄 중분류 5개 조회하시오.

### 4. RegionMaster에서 "강남"이 포함된 지역 5개 조회하시오.

### 5. HotspotAPI에서 update_time이 가장 최신인 데이터 3개 조회하시오.

### 6. HotspotAPI에서 temp가 가장 낮은 데이터 1개 조회하시오.

### 7. CrimeCategory에서 main_cat별로 정렬 후 상위 10개 조회하시오.

### 8. RegionMaster에서 id가 10 이상인 지역 중 5개 조회하시오.

### 9. HotspotAPI에서 ppltn_max 기준 상위 10개 조회하시오.

### 10. CrimeCategory에서 sub_cat 이름이 5글자 이상인 데이터 5개 조회하시오.

## 연습문제 기본2 - 10문제

### 1. CrimeRegion 테이블과 RegionMaster를 JOIN하여 범죄 수가 높은 상위 5개 지역 조회

### 2. CrimeRegion과 CrimeCategory를 JOIN하여 범죄 발생 수 상위 10개 조회

### 3. CrimeTime 테이블에서 특정 범죄(sub_cat="절도")의 시간대별 범죄 상위 5개 조회

### 4. CrimeWeek 테이블에서 가장 범죄 발생이 많은 요일 1개 조회

### 5. RegionMapper를 통해 HotspotAPI와 RegionMaster를 연결하여 혼잡도 높은 상위 5개 지역 조회

### 6. CrimeRegion에서 특정 범죄 category_id의 범죄 발생 상위 3개 지역 조회

### 7. CrimeTime에서 범죄 발생 수가 가장 높은 시간대 1개 조회

### 8. CrimeWeek에서 범죄 발생 상위 5개 요일 조회

### 9. CrimeCategory와 CrimeTime을 JOIN하여 특정 범죄(main_cat="절도")의 시간대별 상위 5개 조회

### 10. CrimeRegion과 RegionMaster를 JOIN하여 "강남구" 관련 데이터 상위 3개 조회